# 03b — Northern BC: connect the proposed IPCAs

Crops the aligned stack to the buffered bounding box of the **4 draft protected areas**
(`proposed_pa_v2_northern_bc.shp`), **locks them in as anchors**, and **up-weights connectivity**
(`transboundary_connectivity` + `climate_corridors` ×5) so the solve identifies the best land to
connect them. Parameters live in `config.ANALYSES["north_bc"]`; outputs →
`output_data/iter6_north_bc/`.

**Note:** the anchors are a large share of the small window, so `budget_pct` must exceed the
locked fraction (see the "locked-in / budget" line in cell 3) — tune it in config if the
feasibility guard stops the run. **Kernel:** `R (y2y)`. Ethan runs cell-by-cell.

In [1]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "north_bc"       # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=north_bc)
prioritizr 8.1.0 | terra 1.9.34 | analysis=north_bc | solver=highs (single solution)
objective=min_shortfall | budget=52% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=hull bounds=[-2003000, 2140000, -1550000, 2679000] (+mask) | lock_in=both
penalties: connectivity=0 | boundary=0 | neighbor=1e-05
outputs -> output_data/iter6_north_bc


In [2]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ROI hull: cropped to 453 x 539 cells + polygon mask
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 453 x 539 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [3]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

Warning message:
“[vect] Z coordinates ignored”


planning units: 178,930 cells | budget = 52% = 93,044 cells
locked-in [existing PAs + lockin_north_bc.gpkg]: 73,397 cells (41.0% of window) -- fits within budget


In [4]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight transboundary_connectivity x5.0 -> 5.0000
  up-weight climate_corridors x5.0 -> 5.0000


In [5]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

neighbor penalty ON (1e-05): binary rook adjacency derived from the PU raster
penalties -> connectivity=0 | boundary=0 | neighbor=1e-05  (0 = off)


In [6]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

Warning message in problem(x, zones(features, zone_names = names(x), feature_names = names(features)), :
“→ `features` has a layer with only zero values.”
A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (178930 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2003000, 2140000, -1550000, 2679000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 93043.6)
│├•penalties: 
││└•1:          neighbor penalties (`penalty` = 0.00001, …)
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 5)
│├•constraints: 
││└•1:          locked in constraints (73397 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      hi

In [7]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

LP has 700807 rows; 529357 cols; 3682444 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [1e-05, 5e+00]

  Bound   [1e+00, 1e+00]

  RHS     [9e+04, 1e+05]



Presolving model

414229 rows, 315505 cols, 2172157 nonzeros 0s

408513 rows, 309784 cols, 922552 nonzeros 1s

Presolve reductions: rows 408513(-292294); columns 309784(-219573); nonzeros 922552(-2759892) 

Solving the presolved LP

IPX model has 408513 rows, 309784 columns and 922552 nonzeros

Input
    Number of variables:                                309784
    Number of free variables:                           0
    Number of constraints:                              408513
    Number of equality constraints:                     0
    Number of matrix entries:                           922552

    Matrix range:                                       [1e+00, 1e+00]

    RHS range:                                          [2e+04, 2e+04]

    Objective range:                                    [1e-05, 3e-03]

In [8]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01    93043.6         52             19644


In [9]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter6_north_bc/portfolio.tif
  output_data/iter6_north_bc/selection_frequency.tif
  output_data/iter6_north_bc/portfolio_representation.csv
  output_data/iter6_north_bc/run_summary.json
